In [1]:
!pip uninstall -y opentelemetry-api opentelemetry-sdk
!pip install -q chromadb sentence-transformers opentelemetry-api opentelemetry-sdk

Found existing installation: opentelemetry-api 1.38.0
Uninstalling opentelemetry-api-1.38.0:
  Successfully uninstalled opentelemetry-api-1.38.0
Found existing installation: opentelemetry-sdk 1.38.0
Uninstalling opentelemetry-sdk-1.38.0:
  Successfully uninstalled opentelemetry-sdk-1.38.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 119.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.1 MB/s eta 0:0

In [2]:
import json
import re
import chromadb
from sentence_transformers import SentenceTransformer

DICT_PATH = "/content/nghe_an_dict_qwen3_cleaned.json"

with open(DICT_PATH, "r", encoding="utf-8") as f:
    dictionary_data = json.load(f)

def norm_text(s):
    return re.sub(r"\s+", " ", str(s).lower().strip())

def build_context(entry):
    parts = []

    meaning = entry.get("meaning", "")
    example = entry.get("example", "")
    example_trans = entry.get("example_translate", "")
    note = entry.get("note", "")
    source_quality = entry.get("source_quality", "")

    if meaning:
        parts.append(f"Nghĩa: {meaning}")
    if example:
        parts.append(f"Ví dụ: {example}")
    if example_trans:
        parts.append(f"Dịch ví dụ: {example_trans}")
    if note:
        parts.append(f"Ghi chú: {note}")
    if source_quality:
        parts.append(f"Độ tin cậy nguồn: {source_quality}")

    return " | ".join(parts)

hash_map = {}

for entry in dictionary_data:
    context_str = build_context(entry)

    keyword = norm_text(entry.get("keyword", ""))
    if keyword:
        hash_map[keyword] = context_str

    for alias in entry.get("aliases", []):
        alias = norm_text(alias)
        if alias:
            hash_map[alias] = context_str

print(f"Hash map built with {len(hash_map)} keys.")

Hash map built with 1070 keys.


In [3]:
model = SentenceTransformer("BAAI/bge-m3")
chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(name="nghe_an_dic")
except Exception:
    pass

collection = chroma_client.get_or_create_collection(name="nghe_an_dic")

documents = []
metadatas = []
ids = []

for i, entry in enumerate(dictionary_data):
    text_to_index = [entry.get("keyword", "")] + entry.get("aliases", [])

    for j, text in enumerate(text_to_index):
        text = norm_text(text)
        if not text:
            continue

        documents.append(text)
        metadatas.append({
            "keyword": entry.get("keyword", ""),
            "definition": entry.get("meaning", ""),
            "example": entry.get("example", ""),
            "example_translate": entry.get("example_translate", ""),
            "note": entry.get("note", ""),
            "source_quality": entry.get("source_quality", "")
        })
        ids.append(f"id_{i}_{j}")

embeddings = model.encode(documents).tolist()

collection.add(
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f"Vector database ready with {len(documents)} entries.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Vector database ready with 1087 entries.


In [4]:
def get_context_with_ngram(text, rag_top_k=3, max_distance=0.35):
    words = re.findall(r"\w+", text.lower(), flags=re.UNICODE)

    found_contexts = []
    found_keys = set()
    matched_indices = set()

    max_n = max(len(k.split()) for k in hash_map.keys())

    for n in range(max_n, 0, -1):
        for i in range(len(words) - n + 1):
            if any(idx in matched_indices for idx in range(i, i + n)):
                continue

            gram = " ".join(words[i:i + n])

            if gram in hash_map:
                found_contexts.append(f"- EXACT {gram.upper()}: {hash_map[gram]}")
                found_keys.add(gram)
                matched_indices.update(range(i, i + n))

    query_emb = model.encode([text]).tolist()
    results = collection.query(
        query_embeddings=query_emb,
        n_results=rag_top_k,
        include=["documents", "metadatas", "distances"]
    )

    for i in range(len(results["documents"][0])):
        match_word = results["documents"][0][i]
        distance = results["distances"][0][i]
        meta = results["metadatas"][0][i]

        if match_word in found_keys:
            continue

        if distance > max_distance:
            continue

        parts = []

        if meta.get("definition"):
            parts.append(f"Nghĩa: {meta['definition']}")
        if meta.get("example"):
            parts.append(f"Ví dụ: {meta['example']}")
        if meta.get("example_translate"):
            parts.append(f"Dịch ví dụ: {meta['example_translate']}")
        if meta.get("note"):
            parts.append(f"Ghi chú: {meta['note']}")
        if meta.get("source_quality"):
            parts.append(f"Độ tin cậy nguồn: {meta['source_quality']}")

        info = " | ".join(parts)
        found_contexts.append(f"- GẦN GIỐNG {match_word.upper()}: {info}")

    return "\n".join(found_contexts)

In [5]:
!pip install openai

In [7]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY"),
)

In [8]:
import re

def clean_translation(text: str) -> str:
    if not text:
        return ""

    text = text.strip()

    if "</think>" in text:
        text = text.split("</think>")[-1].strip()

    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

    prefixes = [
        "Bản dịch:",
        "Tiếng Việt phổ thông:",
        "Câu tiếng Việt phổ thông:",
        "Kết quả:",
    ]

    for prefix in prefixes:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()

    text = text.strip(" \n\r\t\"'“”")

    return text

In [9]:
def translate_nt_text(dialect_text):
    context = get_context_with_ngram(dialect_text)

    messages = [
        {
            "role": "system",
            "content": (
                "Bạn là bộ chuyển đổi phương ngữ Nghệ An/Nghệ Tĩnh sang tiếng Việt phổ thông.\n"
                "Nhiệm vụ: chuyển câu gốc sang tiếng Việt phổ thông tự nhiên, giữ nguyên ý, sắc thái và quan hệ xưng hô.\n"
                "\n"
                "Quy tắc đầu ra:\n"
                "- Chỉ trả về duy nhất bản dịch tiếng Việt phổ thông.\n"
                "- Không giải thích.\n"
                "- Không phân tích.\n"
                "- Không dùng thẻ <think>.\n"
                "- Không lặp lại câu gốc.\n"
                "- Không thêm thông tin mới ngoài ý của câu gốc.\n"
                "\n"
                "Quy tắc chống dịch ảo:\n"
                "- Không được tự sáng tạo từ mới.\n"
                "- Không được thay đổi các từ/cụm từ tiếng Việt phổ thông vốn đã đúng nghĩa.\n"
                "- Nếu một cụm đã là tiếng phổ thông tự nhiên, hãy giữ nguyên cụm đó.\n"
                "- Chỉ thay thế các từ/cụm thật sự là phương ngữ Nghệ An/Nghệ Tĩnh.\n"
                "- Không diễn đạt lại quá xa câu gốc.\n"
                "- Không biến từ phổ thông thành từ khác chỉ vì nghe gần âm.\n"
                "- Ví dụ: 'hậu đậu' phải giữ là 'hậu đậu', không đổi thành 'hôi hồn', 'hú hồn' hay từ khác.\n"
                "- Ví dụ: 'mệt đứt từng khúc ruột' có thể giữ nguyên nếu đã tự nhiên.\n"
                "\n"
                "Quy tắc dùng từ điển/ngữ cảnh:\n"
                "- Ưu tiên các dòng bắt đầu bằng EXACT hơn GẦN GIỐNG.\n"
                "- Chỉ dùng GẦN GIỐNG nếu nó thật sự liên quan trực tiếp đến câu gốc.\n"
                "- Nếu GẦN GIỐNG kéo vào từ tiếng Việt phổ thông như anh, em, tình yêu, lý do, hãy bỏ qua.\n"
                "- Từ điển chỉ là gợi ý, không dịch máy móc từng từ.\n"
                "- Một từ có thể có nhiều nghĩa tùy ngữ cảnh.\n"
                "- Nếu mục có Ghi chú, hãy ưu tiên Ghi chú khi chọn nghĩa.\n"
                "- Nếu Độ tin cậy nguồn là community/needs_review, phải kiểm tra lại bằng ngữ cảnh câu.\n"
                "\n"
                "Quy tắc xử lý đa nghĩa:\n"
                "- Nếu một từ đứng cuối câu hỏi, hãy cân nhắc nó là tiểu từ hỏi như 'không?', 'à?', 'nhỉ?' thay vì đại từ xưng hô.\n"
                "- Nếu một từ đứng ở vị trí chủ ngữ hoặc tân ngữ, hãy cân nhắc nó là đại từ xưng hô như 'cậu', 'bạn', 'nó', 'hắn'.\n"
                "- Với 'ung': nếu đứng cuối câu hỏi thì thường dịch là 'không?'; nếu dùng để gọi người thì dịch là 'cậu/bạn'.\n"
                "- Với 'hấn': thường dịch là 'nó/hắn', nhưng nếu đang nói về sự vật/khái niệm như tình yêu thì có thể dịch là 'nó'.\n"
                "- Với 'rứa/ri/nớ/ni': dịch theo vị trí câu thành 'vậy/thế/này/đó/kia' cho tự nhiên.\n"
                "- Nếu 'ri/rứa' đứng sau tính từ hoặc mô tả trạng thái, thường dịch là 'thế', 'vậy', 'như thế'.\n"
                "- Ví dụ: 'hậu đậu ri' -> 'hậu đậu thế'.\n"
                "- Ví dụ: 'đẹp ri' -> 'đẹp thế'.\n"
                "- Ví dụ: 'ngu rứa' -> 'ngu thế'.\n"
                "\n"
                "Quy tắc xử lý thành ngữ, nói lái và nói trại:\n"
                "- Một số cụm phương ngữ là biến âm, nói lái hoặc nói trại của thành ngữ/cụm từ quen thuộc trong tiếng Việt phổ thông.\n"
                "- Nếu một cụm nghe gần giống một thành ngữ hoặc cách nói quen thuộc, hãy ưu tiên khôi phục về cụm phổ thông tự nhiên nhất.\n"
                "- Không dịch từng chữ nếu cụm đó thực chất là một thành ngữ hoặc cách nói cố định.\n"
                "- Ưu tiên hiểu toàn cụm trước khi hiểu từng từ riêng lẻ.\n"
                "- Nếu dịch từng chữ làm câu mất tự nhiên hoặc vô nghĩa, hãy suy luận theo cụm hoàn chỉnh.\n"
                "- Ví dụ: 'chạy thục mạ' -> 'chạy thục mạng'.\n"
                "- Ví dụ: 'bể chọ' -> 'bể bụng'.\n"
                "- Ví dụ: 'trốc tru' -> 'đầu trâu' hoặc nghĩa bóng là 'ngu'.\n"
                "- Ví dụ: 'cá tràu cá trắm' trong câu chê người có thể mang sắc thái 'trẻ trâu', 'láo láo'.\n"
                "- Ví dụ: 'troèm dứa' trong ngữ cảnh tình cảm có thể mang nghĩa 'tràn đầy'.\n"
                "- Với các cụm có tính thành ngữ, ưu tiên giữ đúng sắc thái cảm xúc thay vì bám sát từng chữ.\n"
                "\n"
                "Quy tắc giữ nguyên tiếng phổ thông:\n"
                "- Không dịch lại các từ vốn đã là tiếng phổ thông nếu chúng không phải phương ngữ trong câu.\n"
                "- Giữ nguyên tên riêng như Nam, địa danh, tên người.\n"
                "- Giữ văn phong tự nhiên, không cần bám sát từng chữ nếu làm câu bị sai nghĩa.\n"
                "- Khi không chắc một từ có phải phương ngữ không, ưu tiên giữ nguyên thay vì tự thay bằng từ khác.\n"
                "\n"
                "Quy tắc chọn mục từ dữ liệu:\n"
                "- Không bắt buộc dùng mọi mục trong từ điển/ngữ cảnh.\n"
                "- Chỉ dùng một mục nếu từ/cụm đó thật sự là phương ngữ trong câu gốc.\n"
                "- Nếu một mục EXACT là từ đơn nhưng nằm trong một cụm tiếng Việt phổ thông, hãy bỏ qua mục đó.\n"
                "- Ví dụ: 'cụ' trong 'cụ thể' không được dịch là 'cậu'.\n"
                "- Ví dụ: 'lại' trong 'em lại thích nó' không được dịch là 'lưỡi'.\n"
                "- Ưu tiên dịch theo cụm dài nhất trước, rồi mới xét từng từ đơn.\n"
                "- Nếu có nhiều nghĩa, chọn nghĩa làm cho cả câu tự nhiên và đúng ngữ pháp nhất.\n"
                "- Nếu nghĩa trong từ điển làm câu vô lý, hãy bỏ nghĩa đó và suy theo ngữ cảnh.\n"
            )
        },
        {
            "role": "user",
            "content": (
                f"Từ điển/ngữ cảnh tham khảo:\n{context}\n\n"
                f"Câu gốc phương ngữ:\n{dialect_text}\n\n"
                "Hãy chuyển sang tiếng Việt phổ thông tự nhiên. Chỉ trả về một bản dịch:"
            )
        }
    ]

    try:
        response = client.chat.completions.create(
            model="deepseek/deepseek-v4-flash",
            messages=messages,
            temperature=0,
            top_p=1,
            max_tokens=512,
            extra_body={
                "reasoning": {
                    "enabled": False
                }
            }
        )

        text = response.choices[0].message.content or ""
        return clean_translation(text)

    except Exception as e:
        print(f"Translate error: {e}")
        return ""

In [12]:
translate_nt_text(f"""
Bựa ni tau đi từ sáng sớm mà mệt đứt từng khúc rọt. Ra ngoài cươi thì bấp cấy cộoc cơn rồi bổ trầy cả trốốc cúi. Mệ tau thấy rứa mới la rằng mi đi đứng kiểu chi mà hậu đậu ri hả. Tau nỏ nói chi, chỉ ngồi một dằm rồi ngó trời ngó đất. Dừ trong nhà đít lác, nác chè cũng hết, cha mệ lại đang đi mần rọong chưa dề. Tau đói quá nên ra nương hái ít rau vô looc ăn tạm.
""")

'Hôm nay tôi đi từ sáng sớm mà mệt đứt từng khúc ruột. Ra ngoài sân thì vấp cái gốc cây rồi ngã trầy cả đầu gối. Mẹ tôi thấy thế mới la rằng mày đi đứng kiểu gì mà hậu đậu thế hả. Tôi không nói gì, chỉ ngồi một chỗ rồi nhìn trời nhìn đất. Bây giờ trong nhà hết tiền, nước uống cũng hết, cha mẹ lại đang đi làm ruộng chưa về. Tôi đói quá nên ra vườn hái ít rau vào luộc ăn tạm.'